# Notebook 06 — Store & Seller Patterns

**Amazon Market Intelligence**  
**Questions answered:** Q5 (Who are my competitors and how do they win?), Q7 (What am I doing wrong?), Q9 (Am I about to get disrupted?)  
**Tool modes served:** Health Check, Competitive Positioning  
**Gold tables:** `gold_store_performance`  

Prior findings to validate at full scale:  
- Finding #7: 55% of sellers have 1 product, 0.4% mega-sellers hold 20%  
- Finding #11: Seller size inversely correlates per-product performance  
- Finding #23: 4.77M stores, 91% Specialist (single category)  
- Finding #24: Specialists earn 44% more per product than Generalists  
- Finding #25: Generalists overrepresented in Kaggle-matched data

## 0 — Setup

In [31]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'amazon_intelligence.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)
TEMPLATE = 'plotly_white'
CHARTS_DIR = 'charts/06_store_seller_patterns'

def save_chart(fig, name, folder=CHARTS_DIR):
    os.makedirs(folder, exist_ok=True)
    fig.write_html(f'{folder}/{name}.html')
    try:
        fig.write_image(f'{folder}/{name}.png', width=1200, height=700, scale=2)
    except Exception as e:
        print(f'PNG failed: {e}')

print(f'Connected to: {DB_PATH}')

Connected to: c:\Users\thinkpad\Desktop\amazon-market-intelligence\data\amazon_intelligence.duckdb


## 1 — Schema Discovery

In [32]:
print('=== gold_store_performance ===')
print(con.sql('DESCRIBE gold_store_performance').df().to_string())
print(f'\nRows: {con.sql("SELECT COUNT(*) FROM gold_store_performance").fetchone()[0]:,}')
con.sql('SELECT * FROM gold_store_performance LIMIT 5').show()

=== gold_store_performance ===
                column_name column_type null   key default extra
0                     store     VARCHAR  YES  None    None  None
1                store_type     VARCHAR  YES  None    None  None
2        ecosystem_products      BIGINT  YES  None    None  None
3            category_count      BIGINT  YES  None    None  None
4               brand_count      BIGINT  YES  None    None  None
5      ecosystem_avg_rating      DOUBLE  YES  None    None  None
6     ecosystem_avg_reviews      DOUBLE  YES  None    None  None
7           kaggle_products      BIGINT  YES  None    None  None
8                 avg_price      DOUBLE  YES  None    None  None
9             total_revenue      DOUBLE  YES  None    None  None
10  avg_revenue_per_product      DOUBLE  YES  None    None  None
11         total_units_sold     HUGEINT  YES  None    None  None
12               pct_active      DOUBLE  YES  None    None  None
13               avg_rating      DOUBLE  YES  None    None 

## 2 — Load Gold Table

In [33]:
df = con.sql('SELECT * FROM gold_store_performance').df()
print(f'Store performance: {len(df):,} rows, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Store performance: 92,656 rows, 15 columns
Columns: ['store', 'store_type', 'ecosystem_products', 'category_count', 'brand_count', 'ecosystem_avg_rating', 'ecosystem_avg_reviews', 'kaggle_products', 'avg_price', 'total_revenue', 'avg_revenue_per_product', 'total_units_sold', 'pct_active', 'avg_rating', 'avg_reviews']


,store,store_type,ecosystem_products,category_count,brand_count,ecosystem_avg_rating,ecosystem_avg_reviews,kaggle_products,avg_price,total_revenue,avg_revenue_per_product,total_units_sold,pct_active,avg_rating,avg_reviews
0,Glock,Generalist,268,9,3,4.28,207.3,1,0.00,0.0,0.00,100.0,100.0,4.60,0.0
1,Muellery,Generalist,148,11,2,3.75,56.1,2,8.34,1233.5,616.75,150.0,100.0,3.45,0.0
2,FORUP,Generalist,121,8,1,4.19,406.9,1,14.99,0.0,0.00,0.0,0.0,4.80,0.0


In [34]:
STORE_COL = 'store'                       
PRODUCT_COUNT_COL = 'kaggle_products'      
ECO_PRODUCTS_COL = 'ecosystem_products'    
TOTAL_REV_COL = 'total_revenue'            
AVG_REV_COL = 'avg_revenue_per_product'    
AVG_PRICE_COL = 'avg_price'               
AVG_RATING_COL = 'avg_rating'             
AVG_REVIEWS_COL = 'avg_reviews'           
CATEGORY_COUNT_COL = 'category_count'     
STORE_TYPE_COL = 'store_type'             
PCT_ACTIVE_COL = 'pct_active'            
BRAND_COUNT_COL = 'brand_count'           
TOTAL_UNITS_COL = 'total_units_sold'      
ECO_RATING_COL = 'ecosystem_avg_rating'  
ECO_REVIEWS_COL = 'ecosystem_avg_reviews'

for name, col in [('STORE', STORE_COL), ('PRODUCTS', PRODUCT_COUNT_COL), ('ECO_PRODUCTS', ECO_PRODUCTS_COL),
                  ('TOTAL_REV', TOTAL_REV_COL), ('AVG_REV', AVG_REV_COL), ('AVG_PRICE', AVG_PRICE_COL),
                  ('AVG_RATING', AVG_RATING_COL), ('AVG_REVIEWS', AVG_REVIEWS_COL),
                  ('CAT_COUNT', CATEGORY_COUNT_COL), ('STORE_TYPE', STORE_TYPE_COL),
                  ('PCT_ACTIVE', PCT_ACTIVE_COL), ('BRAND_COUNT', BRAND_COUNT_COL),
                  ('TOTAL_UNITS', TOTAL_UNITS_COL)]:
    status = '✅' if col in df.columns else '❌ NOT FOUND'
    print(f'{name}: {col} — {status}')

STORE: store — ✅
PRODUCTS: kaggle_products — ✅
ECO_PRODUCTS: ecosystem_products — ✅
TOTAL_REV: total_revenue — ✅
AVG_REV: avg_revenue_per_product — ✅
AVG_PRICE: avg_price — ✅
AVG_RATING: avg_rating — ✅
AVG_REVIEWS: avg_reviews — ✅
CAT_COUNT: category_count — ✅
STORE_TYPE: store_type — ✅
PCT_ACTIVE: pct_active — ✅
BRAND_COUNT: brand_count — ✅
TOTAL_UNITS: total_units_sold — ✅


---

## 3 — Store Size Distribution

Finding #7: 55% of sellers have 1 product, 0.4% mega-sellers hold 20% of revenue.  
How does the long tail look across 92K stores?

### 3.1 — Product Count Distribution

In [35]:
# Quick stats
print(f'Total stores: {len(df):,}')
print(f'Single-product stores: {(df[PRODUCT_COUNT_COL] == 1).sum():,} ({(df[PRODUCT_COUNT_COL] == 1).mean():.1%})')
print(f'Median products/store: {df[PRODUCT_COUNT_COL].median():.0f}')
print(f'Max products/store: {df[PRODUCT_COUNT_COL].max():,}')
print(f'Top 1% threshold: {df[PRODUCT_COUNT_COL].quantile(0.99):.0f}')

fig = px.histogram(
    df[df[PRODUCT_COUNT_COL] <= df[PRODUCT_COUNT_COL].quantile(0.95)],
    x=PRODUCT_COUNT_COL,
    nbins=50,
    title='Store Size Distribution — Product Count (95th percentile cutoff)',
    labels={PRODUCT_COUNT_COL: 'Products per Store'},
    template=TEMPLATE
)
fig.update_layout(yaxis_title='Number of Stores')
save_chart(fig, '01_store_size_distribution')
fig.show()

Total stores: 92,656
Single-product stores: 50,301 (54.3%)
Median products/store: 1
Max products/store: 3,161
Top 1% threshold: 47


### 3.2 — Store Size Tiers

In [36]:
bins = [0, 1, 5, 20, 100, float('inf')]
labels = ['1 product', '2-5 products', '6-20 products', '21-100 products', '100+ products']
df['size_tier'] = pd.cut(df[PRODUCT_COUNT_COL], bins=bins, labels=labels)

tier_stats = (
    df.groupby('size_tier', observed=True)
    .agg(
        store_count=('size_tier', 'size'),
        total_revenue=(TOTAL_REV_COL, 'sum'),
        avg_rev_per_product=(AVG_REV_COL, 'mean'),
        avg_products=(PRODUCT_COUNT_COL, 'mean')
    )
    .reset_index()
)
tier_stats['pct_stores'] = tier_stats['store_count'] / tier_stats['store_count'].sum() * 100
tier_stats['pct_revenue'] = tier_stats['total_revenue'] / tier_stats['total_revenue'].sum() * 100

fig = make_subplots(rows=1, cols=2, subplot_titles=['% of Stores', '% of Revenue'],
                    specs=[[{'type': 'pie'}, {'type': 'pie'}]])

fig.add_trace(go.Pie(labels=tier_stats['size_tier'], values=tier_stats['pct_stores'],
                     hole=0.4, textinfo='label+percent'), row=1, col=1)
fig.add_trace(go.Pie(labels=tier_stats['size_tier'], values=tier_stats['pct_revenue'],
                     hole=0.4, textinfo='label+percent'), row=1, col=2)

fig.update_layout(title='Store Size Tiers — Who Owns the Stores vs Who Owns the Revenue',
                  template=TEMPLATE, height=500, showlegend=False)
save_chart(fig, '02_size_tier_split')
fig.show()

---

## 4 — Specialization vs Diversification

Finding #23: 91% of stores are Specialists (single category).  
Finding #24: Specialists earn 44% more per product.  
Does focus beat breadth?

### 4.1 — Specialist vs Generalist: Revenue per Product

In [37]:
type_stats = (
    df.groupby(STORE_TYPE_COL)
    .agg(
        store_count=(STORE_TYPE_COL, 'size'),
        avg_rev_per_product=(AVG_REV_COL, 'mean'),
        total_revenue=(TOTAL_REV_COL, 'sum'),
        avg_products=(PRODUCT_COUNT_COL, 'mean'),
        avg_rating=(AVG_RATING_COL, 'mean')
    )
    .reset_index()
)
type_stats['pct_stores'] = type_stats['store_count'] / type_stats['store_count'].sum() * 100

print(type_stats.to_string(index=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Avg Revenue per Product ($)', 'Share of Stores (%)'],
                    horizontal_spacing=0.15)

colors = {'Specialist': '#2196F3', 'Focused': '#4CAF50', 'Generalist': '#FF9800'}
for _, row in type_stats.iterrows():
    c = colors.get(row[STORE_TYPE_COL], '#999')
    fig.add_trace(go.Bar(x=[row[STORE_TYPE_COL]], y=[row['avg_rev_per_product']],
                         name=row[STORE_TYPE_COL], marker_color=c,
                         text=[f'${row["avg_rev_per_product"]:,.0f}'], textposition='outside'),
                  row=1, col=1)
    fig.add_trace(go.Bar(x=[row[STORE_TYPE_COL]], y=[row['pct_stores']],
                         marker_color=c, showlegend=False,
                         text=[f'{row["pct_stores"]:.1f}%'], textposition='outside'),
                  row=1, col=2)

fig.update_layout(title='Specialist vs Generalist — Focus Wins',
                  template=TEMPLATE, height=500, showlegend=False)
fig.update_yaxes(title_text='$/product', row=1, col=1)
fig.update_yaxes(title_text='% of stores', row=1, col=2)
save_chart(fig, '03_specialist_vs_generalist')
fig.show()

store_type  store_count  avg_rev_per_product  total_revenue  avg_products  avg_rating  pct_stores
   Focused        30926          4065.526110    306738913.5      2.630537    4.343927   33.377223
Generalist        40329          2902.942074    878262898.0      7.077041    4.331270   43.525514
Specialist        21401          4172.115760    138636268.5      1.782253    4.341799   23.097263


### 4.2 — Category Count vs Per-Product Revenue

In [38]:
cat_perf = (
    df.groupby(CATEGORY_COUNT_COL)
    .agg(
        store_count=(CATEGORY_COUNT_COL, 'size'),
        avg_rev_per_product=(AVG_REV_COL, 'mean'),
        avg_products=(PRODUCT_COUNT_COL, 'mean')
    )
    .reset_index()
)
cat_perf_display = cat_perf[cat_perf[CATEGORY_COUNT_COL] <= 10]

fig = make_subplots(rows=1, cols=2, subplot_titles=['Revenue per Product', 'Number of Stores'],
                    horizontal_spacing=0.12)

fig.add_trace(go.Bar(x=cat_perf_display[CATEGORY_COUNT_COL], y=cat_perf_display['avg_rev_per_product'],
                     marker_color='#E91E63',
                     text=[f'${v:,.0f}' for v in cat_perf_display['avg_rev_per_product']],
                     textposition='outside'), row=1, col=1)

fig.add_trace(go.Bar(x=cat_perf_display[CATEGORY_COUNT_COL], y=cat_perf_display['store_count'],
                     marker_color='#607D8B'), row=1, col=2)

fig.update_layout(title='Category Diversification — More Categories = Less Revenue per Product?',
                  template=TEMPLATE, height=500, showlegend=False)
fig.update_xaxes(title_text='# Categories', dtick=1)
fig.update_yaxes(title_text='Avg $/product', row=1, col=1)
fig.update_yaxes(title_text='Store count', row=1, col=2)
save_chart(fig, '04_category_count_vs_revenue')
fig.show()

---

## 5 — Revenue Concentration

Finding #3: 0.4% of products = 25% of revenue (power law).  
Does the same hold at store level? How concentrated is Amazon's seller economy?

### 5.1 — Top 20 Stores by Revenue

In [39]:
top20 = df.nlargest(20, TOTAL_REV_COL)

fig = px.bar(
    top20,
    x=TOTAL_REV_COL,
    y=STORE_COL,
    orientation='h',
    title='Top 20 Stores by Total Revenue',
    labels={TOTAL_REV_COL: 'Total Revenue ($)', STORE_COL: ''},
    template=TEMPLATE,
    text=[f'${v:,.0f}' for v in top20[TOTAL_REV_COL]]
)
fig.update_traces(textposition='outside')
fig.update_layout(height=700, yaxis={'categoryorder': 'total ascending'})
save_chart(fig, '05_top20_stores')
fig.show()

### 5.2 — Revenue Concentration Curve

In [40]:
df_sorted = df.sort_values(TOTAL_REV_COL, ascending=False).reset_index(drop=True)
df_sorted['cumulative_revenue'] = df_sorted[TOTAL_REV_COL].cumsum()
total_rev = df_sorted[TOTAL_REV_COL].sum()
df_sorted['cum_pct_revenue'] = df_sorted['cumulative_revenue'] / total_rev * 100
df_sorted['cum_pct_stores'] = (df_sorted.index + 1) / len(df_sorted) * 100

for pct in [50, 80, 90]:
    stores_needed = (df_sorted['cum_pct_revenue'] <= pct).sum()
    store_pct = stores_needed / len(df_sorted) * 100
    print(f'{pct}% of revenue comes from top {stores_needed:,} stores ({store_pct:.1f}%)')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_sorted['cum_pct_stores'], y=df_sorted['cum_pct_revenue'],
    mode='lines', line=dict(color='#E91E63', width=3), name='Actual'
))
fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100],
    mode='lines', line=dict(color='gray', dash='dash'), name='Perfect equality'
))
fig.update_layout(
    title='Revenue Concentration Curve — How Unequal is Amazon\'s Seller Economy?',
    xaxis_title='% of Stores (ranked by revenue)',
    yaxis_title='% of Total Revenue',
    template=TEMPLATE, height=600
)
save_chart(fig, '06_revenue_concentration')
fig.show()

50% of revenue comes from top 1,077 stores (1.2%)
80% of revenue comes from top 7,288 stores (7.9%)
90% of revenue comes from top 14,481 stores (15.6%)


---

## 6 — Store Size vs Performance

Finding #11: Seller size inversely correlates per-product performance.  
Bigger stores sell more total — but do they sell more *per product*?

### 6.1 — Products vs Revenue per Product (Scatter)

In [41]:
df_scatter = df[
    (df[PRODUCT_COUNT_COL] <= df[PRODUCT_COUNT_COL].quantile(0.99)) &
    (df[AVG_REV_COL] <= df[AVG_REV_COL].quantile(0.99)) &
    (df[AVG_REV_COL] > 0)
].copy()

fig = px.scatter(
    df_scatter,
    x=PRODUCT_COUNT_COL,
    y=AVG_REV_COL,
    color=STORE_TYPE_COL if STORE_TYPE_COL in df.columns else None,
    opacity=0.3,
    title='Store Size vs Revenue per Product — Does Bigger Mean Better?',
    labels={PRODUCT_COUNT_COL: 'Products in Store', AVG_REV_COL: 'Avg Revenue per Product ($)'},
    template=TEMPLATE,
    log_y=True
)
fig.update_layout(height=600)
save_chart(fig, '07_size_vs_performance')
fig.show()

### 6.2 — Size Tier Performance Comparison

In [42]:
metrics = ['avg_rev_per_product', 'avg_rating', 'avg_products']
metric_labels = ['Revenue per Product ($)', 'Avg Rating', 'Avg Products']

tier_norm = tier_stats.copy()

fig = go.Figure()
for _, row in tier_stats.iterrows():
    fig.add_trace(go.Bar(
        x=[row['size_tier']],
        y=[row['avg_rev_per_product']],
        text=[f'${row["avg_rev_per_product"]:,.0f}'],
        textposition='outside',
        name=str(row['size_tier'])
    ))

fig.update_layout(
    title='Revenue per Product by Store Size Tier',
    xaxis_title='Store Size',
    yaxis_title='Avg Revenue per Product ($)',
    template=TEMPLATE, height=500, showlegend=False
)
save_chart(fig, '08_size_tier_performance')
fig.show()

---

## 7 — Store Rating & Review Patterns

### 7.1 — Store Rating Distribution

In [43]:
fig = px.histogram(
    df[df[AVG_RATING_COL] > 0],
    x=AVG_RATING_COL,
    nbins=50,
    title='Distribution of Average Store Ratings',
    labels={AVG_RATING_COL: 'Average Product Rating'},
    template=TEMPLATE
)
fig.add_vline(x=df[AVG_RATING_COL].median(), line_dash='dash', line_color='red',
              annotation_text=f'Median: {df[AVG_RATING_COL].median():.2f}')
fig.update_layout(yaxis_title='Number of Stores', height=500)
save_chart(fig, '09_store_rating_distribution')
fig.show()

### 7.2 — Rating vs Revenue per Product

In [44]:
df_rated = df[(df[AVG_RATING_COL] > 0) & (df[AVG_REV_COL] > 0)].copy()

df_rated['rating_bin'] = pd.cut(df_rated[AVG_RATING_COL], bins=[0, 2, 3, 3.5, 4, 4.5, 5.01],
                                labels=['<2.0', '2-3', '3-3.5', '3.5-4', '4-4.5', '4.5-5'])

rating_rev = (
    df_rated.groupby('rating_bin', observed=True)
    .agg(
        store_count=('rating_bin', 'size'),
        avg_rev=(AVG_REV_COL, 'mean'),
        median_rev=(AVG_REV_COL, 'median')
    )
    .reset_index()
)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Mean Rev/Product by Rating', 'Store Count by Rating'],
                    horizontal_spacing=0.12)

fig.add_trace(go.Bar(x=rating_rev['rating_bin'], y=rating_rev['avg_rev'],
                     marker_color='#4CAF50',
                     text=[f'${v:,.0f}' for v in rating_rev['avg_rev']],
                     textposition='outside'), row=1, col=1)

fig.add_trace(go.Bar(x=rating_rev['rating_bin'], y=rating_rev['store_count'],
                     marker_color='#2196F3'), row=1, col=2)

fig.update_layout(title='Store Rating vs Revenue — Does Quality Pay?',
                  template=TEMPLATE, height=500, showlegend=False)
save_chart(fig, '10_rating_vs_revenue')
fig.show()

---

## 8 — Key Findings

In [45]:
print('=' * 60)
print('STORE & SELLER PATTERNS — KEY FINDINGS')
print('=' * 60)

print(f'\n1. STORE SIZE DISTRIBUTION:')
single = (df[PRODUCT_COUNT_COL] == 1).mean()
print(f'   Single-product stores: {single:.1%}')
print(f'   Median products/store: {df[PRODUCT_COUNT_COL].median():.0f}')
print(f'   Max products/store: {df[PRODUCT_COUNT_COL].max():,}')

print(f'\n2. SPECIALIZATION:')
if STORE_TYPE_COL in df.columns:
    for _, row in type_stats.iterrows():
        print(f'   {row[STORE_TYPE_COL]}: {row["store_count"]:,} stores ({row["pct_stores"]:.1f}%), '
              f'${row["avg_rev_per_product"]:,.0f}/product')

print(f'\n3. REVENUE CONCENTRATION:')
top1pct_rev = df.nlargest(max(1, len(df)//100), TOTAL_REV_COL)[TOTAL_REV_COL].sum()
total = df[TOTAL_REV_COL].sum()
print(f'   Top 1% of stores hold {top1pct_rev/total:.1%} of revenue')

print(f'\n4. RATING vs REVENUE:')
corr = df[(df[AVG_RATING_COL] > 0) & (df[AVG_REV_COL] > 0)][[AVG_RATING_COL, AVG_REV_COL]].corr().iloc[0,1]
print(f'   Correlation: {corr:.3f}')

STORE & SELLER PATTERNS — KEY FINDINGS

1. STORE SIZE DISTRIBUTION:
   Single-product stores: 54.3%
   Median products/store: 1
   Max products/store: 3,161

2. SPECIALIZATION:
   Focused: 30,926 stores (33.4%), $4,066/product
   Generalist: 40,329 stores (43.5%), $2,903/product
   Specialist: 21,401 stores (23.1%), $4,172/product

3. REVENUE CONCENTRATION:
   Top 1% of stores hold 47.8% of revenue

4. RATING vs REVENUE:
   Correlation: 0.036


In [46]:
con.close()
print('Done. DuckDB connection closed.')
print(f'Charts saved to: {CHARTS_DIR}/')

Done. DuckDB connection closed.
Charts saved to: charts/06_store_seller_patterns/
